# Pakistan Gender & Development Analysis

## 01 — Data Collection

This notebook collects gender, education, economic, and demographic indicators for Pakistan from the World Bank World Development Indicators (WDI).

### Research Question

How are education, economic conditions, and demographic factors associated with female labor force participation in Pakistan?

### Data Source

World Bank — World Development Indicators (WDI)

### Country

Pakistan

### Main Outcome Variable

Female labor force participation rate (% of female population ages 15+)

In [1]:
import pandas as pd
import requests
import io

In [2]:
indicators = {
    "Female_Labor_Force_Participation": "SL.TLF.CACT.FE.ZS",
    "Female_Secondary_Enrollment": "SE.SEC.ENRR.FE",
    "Female_Literacy": "SE.ADT.LITR.FE.ZS",
    "GDP_per_Capita": "NY.GDP.PCAP.CD",
    "Fertility_Rate": "SP.DYN.TFRT.IN",
    "Female_Unemployment": "SL.UEM.TOTL.FE.ZS",
    "Urban_Population": "SP.URB.TOTL.IN.ZS",
    "Life_Expectancy": "SP.DYN.LE00.IN",
    "Access_to_Electricity": "EG.ELC.ACCS.ZS"
}

country = "PAK"

In [3]:
all_data = []

for variable, indicator_code in indicators.items():

    url = (
        f"https://api.worldbank.org/v2/country/{country}/indicator/"
        f"{indicator_code}?format=json&per_page=100"
    )

    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()[1]

        for observation in data:
            all_data.append({
                "Country": observation["country"]["value"],
                "Country_Code": observation["countryiso3code"],
                "Year": int(observation["date"]),
                "Indicator": variable,
                "Value": observation["value"]
            })
    else:
        print(f"Failed to retrieve {variable}")

In [4]:
df_long = pd.DataFrame(all_data)

df_long.head()

,Country,Country_Code,Year,Indicator,Value
0,Pakistan,PAK,2025,Female_Labor_Force_Participation,24.046
1,Pakistan,PAK,2024,Female_Labor_Force_Participation,23.989
2,Pakistan,PAK,2023,Female_Labor_Force_Participation,23.997
3,Pakistan,PAK,2022,Female_Labor_Force_Participation,24.027
4,Pakistan,PAK,2021,Female_Labor_Force_Participation,24.365


In [5]:
df = df_long.pivot(
    index=["Country", "Country_Code", "Year"],
    columns="Indicator",
    values="Value"
).reset_index()

df.head()

Indicator,Country,Country_Code,Year,Access_to_Electricity,Female_Labor_Force_Participation,Female_Literacy,Female_Secondary_Enrollment,Female_Unemployment,Fertility_Rate,GDP_per_Capita,Life_Expectancy,Urban_Population
0,Pakistan,PAK,1960,NaN,NaN,NaN,NaN,NaN,6.8,82.024100,44.101,22.299698
1,Pakistan,PAK,1961,NaN,NaN,NaN,NaN,NaN,6.8,87.777825,44.956,22.632127
2,Pakistan,PAK,1962,NaN,NaN,NaN,NaN,NaN,6.8,89.503953,45.850,22.934215
3,Pakistan,PAK,1963,NaN,NaN,NaN,NaN,NaN,6.8,93.650873,46.761,23.208181
4,Pakistan,PAK,1964,NaN,NaN,NaN,NaN,NaN,6.8,102.459764,47.831,23.460517


In [6]:
df = df.sort_values("Year").reset_index(drop=True)

df.head()

Indicator,Country,Country_Code,Year,Access_to_Electricity,Female_Labor_Force_Participation,Female_Literacy,Female_Secondary_Enrollment,Female_Unemployment,Fertility_Rate,GDP_per_Capita,Life_Expectancy,Urban_Population
0,Pakistan,PAK,1960,NaN,NaN,NaN,NaN,NaN,6.8,82.024100,44.101,22.299698
1,Pakistan,PAK,1961,NaN,NaN,NaN,NaN,NaN,6.8,87.777825,44.956,22.632127
2,Pakistan,PAK,1962,NaN,NaN,NaN,NaN,NaN,6.8,89.503953,45.850,22.934215
3,Pakistan,PAK,1963,NaN,NaN,NaN,NaN,NaN,6.8,93.650873,46.761,23.208181
4,Pakistan,PAK,1964,NaN,NaN,NaN,NaN,NaN,6.8,102.459764,47.831,23.460517


In [7]:
print("Dataset shape:", df.shape)

print("\nYear range:")
print(df["Year"].min(), "to", df["Year"].max())

print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (66, 12)

Year range:
1960 to 2025

Columns:
['Country', 'Country_Code', 'Year', 'Access_to_Electricity', 'Female_Labor_Force_Participation', 'Female_Literacy', 'Female_Secondary_Enrollment', 'Female_Unemployment', 'Fertility_Rate', 'GDP_per_Capita', 'Life_Expectancy', 'Urban_Population']


In [8]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": df.isna().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "Missing_Percentage",
    ascending=False
)

missing_summary

,Missing_Count,Missing_Percentage
Indicator,,
Female_Literacy,50,75.757576
Access_to_Electricity,39,59.090909
Female_Unemployment,31,46.969697
Female_Labor_Force_Participation,30,45.454545
Female_Secondary_Enrollment,26,39.393939
Fertility_Rate,1,1.515152
Life_Expectancy,1,1.515152
Country,0,0.000000
Year,0,0.000000


In [9]:
df.to_csv("pakistan_gender_development_raw.csv", index=False)

print("Raw dataset saved successfully.")

Raw dataset saved successfully.
